# 📊 HPO Results — Master Summary

Aggregates every saved best-param artifact under `notebooks/hpo/results/best/` (classical, deep learning, and quantum). Run the model notebooks first; this notebook only reads their outputs, so it can be re-run any time to refresh the leaderboard.


In [ ]:
import sys
import os

cwd = os.path.abspath(os.getcwd())
project_root = cwd.split("quantum-multiclass-classification")[0] + "quantum-multiclass-classification"
print(f"Project root: {project_root}")

sys.path.append(os.path.abspath(project_root))

from utils.prepare_data import prepare_data


In [ ]:
import json
import pandas as pd
from pathlib import Path

BEST_DIR = Path(project_root) / "notebooks" / "hpo" / "results" / "best"
files = sorted(BEST_DIR.glob("*.json"))
print(f"Found {len(files)} best-param artifacts in {BEST_DIR}")

rows = []
for fp in files:
    with open(fp, encoding="utf-8") as f:
        r = json.load(f)
    params_str = ' | '.join(f'{k}={v}' for k, v in r.get('params', {}).items())
    rows.append({
        'Category'       : r.get('category'),
        'Family'         : r.get('family', ''),
        'Circuit/Kernel' : r.get('circuit', ''),
        'Model'          : r.get('model'),
        'Accuracy'       : r.get('acc'),
        'Precision'      : r.get('prec'),
        'Recall'         : r.get('rec'),
        'F1-Score'       : r.get('f1'),
        'F1-Macro'       : r.get('f1_macro'),
        'ROC-AUC'        : r.get('roc'),
        'PR-AUC'         : r.get('pra'),
        'Log-Loss'       : r.get('loss'),
        'Selection'      : r.get('selection_score'),
        'Loss Gap'       : r.get('loss_gap'),
        'F1 Gap'         : r.get('f1_gap'),
        'Fit'            : r.get('fit_verdict', ''),
        'Train Acc'      : r.get('train_acc'),
        'Val Acc'        : r.get('val_acc'),
        'Fit Gap'        : r.get('fit_gap'),
        'Exec. Time (s)' : round(r['execution_time'], 2) if r.get('execution_time') is not None else None,
        'Best Params'    : params_str,
        'Artifact'       : fp.name,
    })

all_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)
print(f"Total evaluations: {len(all_df)}")
all_df.head()


## Combined leaderboard


In [ ]:
# Full ranking by selection score (penalized val log-loss, lower = better)
if len(all_df):
    ranked = all_df.sort_values('Selection', ascending=True).reset_index(drop=True)
    ranked.index += 1
    out = Path(project_root) / "notebooks" / "hpo" / "results" / "eval_all_final_hpo.csv"
    ranked.to_csv(out, index=True)
    print(f"✅ Saved combined ranking: {out}")
    display(ranked)
else:
    print("No artifacts yet — run the model notebooks first.")


## Per-category leaderboards


In [ ]:
# Per-category leaderboards
for cat, g in all_df.groupby('Category'):
    print(f"\n===== {cat.upper()} ({len(g)} models) =====")
    gg = g.sort_values('Selection', ascending=True).reset_index(drop=True)
    gg.index += 1
    display(gg[['Family', 'Circuit/Kernel', 'Model', 'Accuracy', 'F1-Macro',
                'Log-Loss', 'Selection', 'Fit', 'Best Params']])


## Summary plots


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(all_df):
    # 1) Selection score per model (top 30 = lowest penalized log-loss), by category
    plot_df = all_df.dropna(subset=['Selection']).sort_values('Selection', ascending=False)
    if len(plot_df) > 30:
        plot_df = plot_df.tail(30)   # keep the 30 best (lowest selection)
    cats = plot_df['Category'].astype('category')
    palette = {c: clr for c, clr in zip(cats.cat.categories,
               plt.cm.tab10.colors)}
    colors = [palette[c] for c in plot_df['Category']]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.32 * len(plot_df))))
    ax.barh(plot_df['Model'], plot_df['Selection'].astype(float), color=colors)
    ax.set_xlabel('Selection = penalized val log-loss (lower = better)')
    ax.set_title('Best selection score per model (top 30, lower = better)')
    handles = [plt.Rectangle((0, 0), 1, 1, color=palette[c]) for c in palette]
    ax.legend(handles, list(palette.keys()), title='Category')
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    fig.savefig(Path(project_root) / "notebooks" / "hpo" / "results" / "summary_selection.png",
                dpi=120, bbox_inches='tight')
    plt.show()

    # 2) Fit verdict counts
    fig, ax = plt.subplots(figsize=(6, 4))
    vc = all_df['Fit'].value_counts()
    ax.bar(vc.index, vc.values, color=['#55A868', '#C44E52', '#8172B2', '#999999'][:len(vc)])
    ax.set_title('Best-param fit diagnosis across all models')
    ax.set_ylabel('Count')
    for i, v in enumerate(vc.values):
        ax.text(i, v, str(v), ha='center', va='bottom')
    fig.tight_layout()
    fig.savefig(Path(project_root) / "notebooks" / "hpo" / "results" / "summary_fit.png",
                dpi=120, bbox_inches='tight')
    plt.show()

    # 3) Train vs Val accuracy scatter (generalization view)
    sub = all_df.dropna(subset=['Train Acc', 'Val Acc'])
    if len(sub):
        fig, ax = plt.subplots(figsize=(6, 6))
        for c in sub['Category'].unique():
            s = sub[sub['Category'] == c]
            ax.scatter(s['Train Acc'].astype(float), s['Val Acc'].astype(float),
                       label=c, alpha=0.8)
        lim = [0.0, 1.02]
        ax.plot(lim, lim, 'k--', lw=1, alpha=0.5)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel('Train Accuracy'); ax.set_ylabel('Validation Accuracy')
        ax.set_title('Train vs Validation (points below diagonal = overfit)')
        ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(Path(project_root) / "notebooks" / "hpo" / "results" / "summary_trainval.png",
                    dpi=120, bbox_inches='tight')
        plt.show()
